In [11]:
import anthropic
import os

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue


from langsmith import Client

import json

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

### Get all recods from the qdrant

In [3]:
all_points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-00",
        limit=100,
        offset=None,
        with_payload=True,
        with_vectors=False
)

In [7]:
all_points[0][0].payload

{'description': "USB C Hub, MCY USB C to HDMI Multiptort Adapter, 10 in 1 Portable Type C Dongle with 4K HDMI, VGA, PD Charging, USB 3.0 and USB 2.0 Ports, SD/TF Card, Compatible with MacBook, HP, Dell and More 【10 IN 1 USB C Docking Station】MCY Type-C smart docking station can solve 10 kinds of office troubles at one time. USB C hub multiport adapter hub rich interface. HDMI makes the big screen clearer. 1G files are transferred in 3 seconds, and there is no need to wait for the transfer. 【Wonderful more than one side】10-port USB C to HDMI adapter turns your laptop into a thin and light desktop computer. It is equipped with 4K@60Hz HDMI port, VGA port, SD/TF card reader and USB 3.0 and USB 2.0 ports. Get rid of the problem of insufficient notebook ports and connect more devices. A Type C interface helps you upgrade your space. 【4K Video HDMI to USB C Hub】Mirror or extend your laptop/tablet/phone's screen with hdmi port and directly stream 4K@60Hz (3840 x 2160) or full HD 1080P lifelik

### use only description and parent_asin

In [8]:
all_context = [
        {
                "id": data.payload["parent_asin"], 
                "text": data.payload["description"]
        }
        for data in all_points[0]
]

In [9]:
all_context[0:2]

[{'id': 'B0BTYK7SB3',
  'text': "USB C Hub, MCY USB C to HDMI Multiptort Adapter, 10 in 1 Portable Type C Dongle with 4K HDMI, VGA, PD Charging, USB 3.0 and USB 2.0 Ports, SD/TF Card, Compatible with MacBook, HP, Dell and More 【10 IN 1 USB C Docking Station】MCY Type-C smart docking station can solve 10 kinds of office troubles at one time. USB C hub multiport adapter hub rich interface. HDMI makes the big screen clearer. 1G files are transferred in 3 seconds, and there is no need to wait for the transfer. 【Wonderful more than one side】10-port USB C to HDMI adapter turns your laptop into a thin and light desktop computer. It is equipped with 4K@60Hz HDMI port, VGA port, SD/TF card reader and USB 3.0 and USB 2.0 ports. Get rid of the problem of insufficient notebook ports and connect more devices. A Type C interface helps you upgrade your space. 【4K Video HDMI to USB C Hub】Mirror or extend your laptop/tablet/phone's screen with hdmi port and directly stream 4K@60Hz (3840 x 2160) or full 

### Use LLM to generate synthetic Eval refence dataset

In [13]:
# use structured outputs, right now we are doing it without it

output_schema = {
        "type": "array",
        "items": {
                "type": "object",
                "properties": {
                        "question": {
                                "type": "string",
                                "description": "Suggested question."
                        },
                        "chunk_ids": {
                                "type": "array",
                                "items": {
                                        "type": "string",
                                        "description": "ID of the chunk that colud be used to answer the question."
                                }
                        },
                        "answer_example": {
                                "type": "string",
                                "description": "Suggested answer grounded in the context."
                        },
                        "reasoning": {
                                "type": "string",
                                "description": "Reasoning why the question could be answered with the chunks."
                        }
                }
        }
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
I want you to come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{all_context}
"""


In [16]:
print(USER_PROMPT)


Here is the list of chunks, each list element is a dictionary with id and text:
[{'id': 'B0BTYK7SB3', 'text': "USB C Hub, MCY USB C to HDMI Multiptort Adapter, 10 in 1 Portable Type C Dongle with 4K HDMI, VGA, PD Charging, USB 3.0 and USB 2.0 Ports, SD/TF Card, Compatible with MacBook, HP, Dell and More 【10 IN 1 USB C Docking Station】MCY Type-C smart docking station can solve 10 kinds of office troubles at one time. USB C hub multiport adapter hub rich interface. HDMI makes the big screen clearer. 1G files are transferred in 3 seconds, and there is no need to wait for the transfer. 【Wonderful more than one side】10-port USB C to HDMI adapter turns your laptop into a thin and light desktop computer. It is equipped with 4K@60Hz HDMI port, VGA port, SD/TF card reader and USB 3.0 and USB 2.0 ports. Get rid of the problem of insufficient notebook ports and connect more devices. A Type C interface helps you upgrade your space. 【4K Video HDMI to USB C Hub】Mirror or extend your laptop/tablet/p

In [20]:
client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=8096,
    system=SYSTEM_PROMPT,
    messages=[
        {"role": "user", "content": USER_PROMPT}
    ]
)

raw_text = response.content[0].text
print(raw_text)

```json
[
  {
    "question": "What are all the audio devices you have available for listening to music?",
    "chunk_ids": ["B09ZWNVYV2", "B0BJ9PRHZ3", "B09MQBZC2B", "B0C5RB75WB", "B09F36P17Y", "B0BS1GQJ5S", "B0BG29CCQ3", "B09TKF5S6W", "B0C32SMPNP"],
    "answer_example": "We have several audio devices available: JBL GO 3 waterproof Bluetooth speaker, Soundcore Space A40 noise-cancelling earbuds with 50-hour playtime, NYANDU wireless earbuds with 40-hour playtime, HCMOBI bone conduction headphones with 8-10 hour battery life, USB C headphones for devices without 3.5mm jacks, and multiple true wireless earbuds options.",
    "reasoning": "Multiple chunks contain information about different audio devices. Combining all audio product chunks provides a comprehensive overview of available listening devices.",
    "category": "multiple_chunks"
  },
  {
    "question": "I want to set up a complete workspace at home. What products do you have that could help me build this?",
    "chunk_ids": 

In [23]:
import re
cleaned = re.sub(r"^```json\s*|\s*```$", "", raw_text.strip())
eval_dataset = json.loads(cleaned)
eval_dataset[0]

{'question': 'What are all the audio devices you have available for listening to music?',
 'chunk_ids': ['B09ZWNVYV2',
  'B0BJ9PRHZ3',
  'B09MQBZC2B',
  'B0C5RB75WB',
  'B09F36P17Y',
  'B0BS1GQJ5S',
  'B0BG29CCQ3',
  'B09TKF5S6W',
  'B0C32SMPNP'],
 'answer_example': 'We have several audio devices available: JBL GO 3 waterproof Bluetooth speaker, Soundcore Space A40 noise-cancelling earbuds with 50-hour playtime, NYANDU wireless earbuds with 40-hour playtime, HCMOBI bone conduction headphones with 8-10 hour battery life, USB C headphones for devices without 3.5mm jacks, and multiple true wireless earbuds options.',
 'reasoning': 'Multiple chunks contain information about different audio devices. Combining all audio product chunks provides a comprehensive overview of available listening devices.',
 'category': 'multiple_chunks'}

In [ ]:
# llm dont provide exact number, we asked for 30 we got 33
len(eval_dataset)

33

In [49]:
def get_description(parent_asin: str) -> str:
        points = qdrant_client.scroll(
                collection_name="Amazon-items-collection-00",
                limit=100,
                with_payload=True,
                with_vectors=False,
                scroll_filter=Filter(
                        must=[
                                FieldCondition(
                                        key="parent_asin",
                                        match=MatchValue(value=parent_asin)
                                )
                        ]
                )
        )
        return points[0][0].payload["description"]


### Create Eval dataset in Langsmith

In [26]:
client = Client(api_key=os.environ["LANGSMITH_API_KEY"])

In [31]:
dataset = client.create_dataset(
        dataset_name="rag-evaluation-dataset",
        description="Dataset for evaluating RAG pipeline"
)

In [50]:
for item in eval_dataset:
        client.create_example(
                dataset_id=dataset.id,
                inputs={"question": item["question"]},
                outputs={
                        "ground_truth": item["answer_example"],
                        "reference_context_ids": item["chunk_ids"],
                        "reference_descriptions": [get_description(parent_asin) for parent_asin in item["chunk_ids"]],
                }
        )